In [1]:
# load libraies

%run py_libraries.py

/Users/4476224/.local/lib/python3.8/site-packages/tensorflow_addons/utils/ensure_tf_install.py:53: UserWarning: Tensorflow Addons supports using Python ops for all Tensorflow versions above or equal to 2.6.0 and strictly below 2.9.0 (nightly versions are not supported). 
 The versions of TensorFlow you are currently using is 2.13.0 and is not supported. 
Some things might work, some things might not.
If you were to encounter a bug, do not file an issue.
If you want to make sure you're using a tested and supported configuration, either change the TensorFlow version or the TensorFlow Addons's version. 
You can find the compatibility matrix in TensorFlow Addon's readme:
https://github.com/tensorflow/addons
  warnings.warn(


Instructions for updating:
non-resource variables are not supported in the long term


In [2]:
# loading utility files

from utility.sv_fig import savefig
from utility.make_cm import make_confusion_matrix
from utility.get_g_result import get_gamma_results
from utility.plt_result import plot_results

In [3]:
def savefig(filename, crop = True):
    plt.savefig('{}.pdf'.format(filename))

In [4]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_row', None)

# load data
X_train = pd.read_csv('data/norm_X_train_Yes_No.csv')
X_test = pd.read_csv('data/norm_X_test_Yes_No.csv')
y_train = pd.read_csv('data/y_train_Yes_No.csv')
y_test = pd.read_csv('data/y_test_Yes_No.csv')


print(X_test.shape)

(32, 15)


In [5]:
# Print the column names as a list
print(X_test.columns.tolist())

['Age_at_Surgery', 'Race', 'Surgery', 'Smoker', 'BMI', 'NAC', 'cT', 'pT', 'cT_or_pT', 'pN', 'Bx_Histology', 'Histology', 'Sample_weight_g_tumor', 'Tumor_digest_count_primary_tumor', 'Number_of_fragments_plated_tumor']


In [6]:
Xy_Yes_No = pd.concat([X_test, y_test], axis=1)

In [7]:
# checking number of patients not missing data

ndata_f_noMiss = Xy_Yes_No.copy()
num_complete_rows = ndata_f_noMiss.dropna().shape[0]
print(num_complete_rows)

18


In [7]:
#
col_to_scale = ['Age_at_Surgery', 'BMI', 'Sample_weight_g_tumor', 'Tumor_digest_count_primary_tumor',]

def midas_impl(missing, col_to_scale, n_epoch):
    missing_cp = missing.copy() # make copy of missing df
    
    # scaler = MinMaxScaler().fit(missing_cp[col_to_scale]) # scale data
    scaler = StandardScaler().fit(missing_cp[col_to_scale]) # scale data
    missing_sc = scaler.transform(missing_cp[col_to_scale]) # this is an array, convert to df
    # missing_sc = missing_cp.copy()

    missing_df = pd.DataFrame({'Age_at_Surgery':missing_sc[:,0],
                               'BMI':missing_sc[:,1],
                               'Sample_weight_g_tumor':missing_sc[:,2],
                               'Tumor_digest_count_primary_tumor':missing_sc[:,3],
                               })
    
    # Extract cat columns from missing
    ext_col1 = missing_cp['Race']
    ext_col2 = missing_cp['Surgery']
    ext_col3 = missing_cp['Smoker']
    ext_col4 = missing_cp['NAC']
    ext_col5 = missing_cp['cT']
    ext_col6 = missing_cp['pT']
    ext_col7 = missing_cp['cT_or_pT']
    ext_col8 = missing_cp['pN']
    ext_col9 = missing_cp['Bx_Histology']
    ext_col10 = missing_cp['Histology']
    ext_col11 = missing_cp['Number_of_fragments_plated_tumor']
    ext_col12 = missing_cp['OverallTILGrowth']
    

    # Add the extracted cat columns to missing_df
    missing_df = pd.concat([missing_df, ext_col1.rename('Race')], axis=1)
    missing_df = pd.concat([missing_df, ext_col2.rename('Surgery')], axis=1)
    missing_df = pd.concat([missing_df, ext_col3.rename('Smoker')], axis=1)
    missing_df = pd.concat([missing_df, ext_col4.rename('NAC')], axis=1)
    missing_df = pd.concat([missing_df, ext_col5.rename('cT')], axis=1)
    missing_df = pd.concat([missing_df, ext_col6.rename('pT')], axis=1)
    missing_df = pd.concat([missing_df, ext_col7.rename('cT_or_pT')], axis=1)
    missing_df = pd.concat([missing_df, ext_col8.rename('pN')], axis=1)
    missing_df = pd.concat([missing_df, ext_col9.rename('Bx_Histology')], axis=1)
    missing_df = pd.concat([missing_df, ext_col10.rename('Histology')], axis=1)
    missing_df = pd.concat([missing_df, ext_col11.rename('Number_of_fragments_plated_tumor')], axis=1)
    missing_df = pd.concat([missing_df, ext_col12.rename('OverallTILGrowth')], axis=1)
    

    missing_df.to_csv('missing_test.csv')
    data_0 = pd.read_csv('missing_test.csv')
    data_0.columns.str.strip()

    categorical = ['Race','Surgery','Smoker','NAC','cT','pT','cT_or_pT','pN','Bx_Histology','Histology','Number_of_fragments_plated_tumor','OverallTILGrowth']
    
    data_cat, cat_cols_list = md.cat_conv(data_0[categorical])

    data_0.drop(categorical, axis = 1, inplace = True)
    constructor_list = [data_0]
    constructor_list.append(data_cat)
    data_in = pd.concat(constructor_list, axis=1)

    na_loc = data_in.isnull()
    data_in[na_loc] = np.nan

    # imputer = md.Midas(layer_structure = [128,128], vae_layer = True, seed = 89, input_drop = 0.75, learn_rate = 0.0001)
    imputer = md.Midas(layer_structure = [128,128], vae_layer = False, seed = 42, input_drop = 0.75, learn_rate = 0.001)
    imputer.build_model(data_in, softmax_columns = cat_cols_list)
    imputer.train_model(training_epochs = n_epoch)

    # do not update M
    imputations = imputer.generate_samples(m=20).output_list

    flat_cats = [cat for variable in cat_cols_list for cat in variable]
    for i in range(len(imputations)):
        tmp_cat = [imputations[i][x].idxmax(axis=1) for x in cat_cols_list]
        cat_df = pd.DataFrame({categorical[i]:tmp_cat[i] for i in range(len(categorical))})
        imputations[i] = pd.concat([imputations[i], cat_df], axis = 1).drop(flat_cats, axis = 1)

    # Implementing an efficient way to replace 'Column_name_Num' with Num in every data_item whose column is 'Column_name'    
    # Loop through all imputations
    for i in range(len(imputations)):
        df = imputations[i]
        
        # Process each column independently
        for col in df.columns:
            
            # Only operate on string/object columns
            if df[col].dtype == 'object':
                # Pattern: match "ColumnName_Number"
                pattern = rf'^{re.escape(col)}_(.+)$'
                # pattern = rf'^{re.escape(col)}_(\d+)$'
                
                df[col] = df[col].str.replace(
                    pattern,
                    r'\1',
                    regex=True
                )
                
        # Optional: convert to numeric if desired
        df = df.apply(pd.to_numeric, errors='ignore')
        imputations[i] = df

    # rescale imputations
    imputations[0][col_to_scale] = scaler.inverse_transform(imputations[0][col_to_scale])
    imputations[1][col_to_scale] = scaler.inverse_transform(imputations[1][col_to_scale])
    imputations[2][col_to_scale] = scaler.inverse_transform(imputations[2][col_to_scale])
    imputations[3][col_to_scale] = scaler.inverse_transform(imputations[3][col_to_scale])
    imputations[4][col_to_scale] = scaler.inverse_transform(imputations[4][col_to_scale])
    imputations[5][col_to_scale] = scaler.inverse_transform(imputations[5][col_to_scale])
    imputations[6][col_to_scale] = scaler.inverse_transform(imputations[6][col_to_scale])
    imputations[7][col_to_scale] = scaler.inverse_transform(imputations[7][col_to_scale])
    imputations[8][col_to_scale] = scaler.inverse_transform(imputations[8][col_to_scale])
    imputations[9][col_to_scale] = scaler.inverse_transform(imputations[9][col_to_scale])
    imputations[10][col_to_scale] = scaler.inverse_transform(imputations[10][col_to_scale])
    imputations[11][col_to_scale] = scaler.inverse_transform(imputations[11][col_to_scale])
    imputations[12][col_to_scale] = scaler.inverse_transform(imputations[12][col_to_scale])
    imputations[13][col_to_scale] = scaler.inverse_transform(imputations[13][col_to_scale])
    imputations[14][col_to_scale] = scaler.inverse_transform(imputations[14][col_to_scale])
    imputations[15][col_to_scale] = scaler.inverse_transform(imputations[15][col_to_scale])
    imputations[16][col_to_scale] = scaler.inverse_transform(imputations[16][col_to_scale])
    imputations[17][col_to_scale] = scaler.inverse_transform(imputations[17][col_to_scale])
    imputations[18][col_to_scale] = scaler.inverse_transform(imputations[18][col_to_scale])
    imputations[19][col_to_scale] = scaler.inverse_transform(imputations[19][col_to_scale])

    n=0
    for i in imputations:
        file_out_csv = "midas_test/midas_test_" + str(n) + ".csv"
        i.to_csv(file_out_csv, index=False)
        n += 1

    df0 = pd.read_csv('midas_test/midas_test_0.csv'); df0 = df0.iloc[:,1:]
    df1 = pd.read_csv('midas_test/midas_test_1.csv'); df1 = df1.iloc[:,1:]
    df2 = pd.read_csv('midas_test/midas_test_2.csv'); df2 = df2.iloc[:,1:]
    df3 = pd.read_csv('midas_test/midas_test_3.csv'); df3 = df3.iloc[:,1:]
    df4 = pd.read_csv('midas_test/midas_test_4.csv'); df4 = df4.iloc[:,1:]
    df5 = pd.read_csv('midas_test/midas_test_5.csv'); df5 = df5.iloc[:,1:]
    df6 = pd.read_csv('midas_test/midas_test_6.csv'); df6 = df6.iloc[:,1:]
    df7 = pd.read_csv('midas_test/midas_test_7.csv'); df7 = df7.iloc[:,1:]
    df8 = pd.read_csv('midas_test/midas_test_8.csv'); df8 = df8.iloc[:,1:]
    df9 = pd.read_csv('midas_test/midas_test_9.csv'); df9 = df9.iloc[:,1:]
    df10 = pd.read_csv('midas_test/midas_test_10.csv'); df10 = df10.iloc[:,1:]
    df11 = pd.read_csv('midas_test/midas_test_11.csv'); df11 = df11.iloc[:,1:]
    df12 = pd.read_csv('midas_test/midas_test_12.csv'); df12 = df12.iloc[:,1:]
    df13 = pd.read_csv('midas_test/midas_test_13.csv'); df13 = df13.iloc[:,1:]
    df14 = pd.read_csv('midas_test/midas_test_14.csv'); df14 = df14.iloc[:,1:]
    df15 = pd.read_csv('midas_test/midas_test_15.csv'); df15 = df15.iloc[:,1:]
    df16 = pd.read_csv('midas_test/midas_test_16.csv'); df16 = df16.iloc[:,1:]
    df17 = pd.read_csv('midas_test/midas_test_17.csv'); df17 = df17.iloc[:,1:]
    df18 = pd.read_csv('midas_test/midas_test_18.csv'); df18 = df18.iloc[:,1:]
    df19 = pd.read_csv('midas_test/midas_test_19.csv'); df19 = df19.iloc[:,1:]

    # Initialize the output dataframe with the same dimensions
    output_df = pd.DataFrame(index=df0.index, columns=df0.columns)

    # Process continuous columns
    for col in df1.columns[:4]:
        output_df[col] = (abs(df0[col]) + abs(df1[col]) + abs(df2[col]) + abs(df3[col]) + abs(df4[col]) + 
                          abs(df5[col]) + abs(df6[col]) + abs(df7[col]) + abs(df8[col]) + abs(df9[col]) + 
                         abs(df10[col]) + abs(df11[col]) + abs(df12[col]) + abs(df13[col]) + abs(df14[col]) + 
                          abs(df15[col]) + abs(df16[col]) + abs(df17[col]) + abs(df18[col]) + abs(df19[col])) / 20

    # Process categorical columns
    for col in df1.columns[4:]:
        output_df[col] = pd.concat([df0[col],df1[col],df2[col],df3[col],df4[col],df5[col],df6[col],df7[col],df8[col],df9[col],
                                   df10[col],df11[col],df12[col],df13[col],df14[col],df15[col],df16[col],df17[col],df18[col],df19[col]], axis=1).median(axis=1)

    # Write the output dataframe to a CSV file
    output_df.to_csv('output_test.csv', index=False)

    # re-order cols in output_df
    output_df_re = output_df[['Age_at_Surgery', 'Race', 'Surgery', 'Smoker', 'BMI', 'NAC', 'cT', 'pT', 'cT_or_pT', 'pN', 
                              'Bx_Histology', 'Histology', 'Sample_weight_g_tumor', 'Tumor_digest_count_primary_tumor', 
                              'Number_of_fragments_plated_tumor','OverallTILGrowth']].copy()
    return output_df_re





In [8]:
# calculate proportion of missingness in ndata_noR

# total num of NaN in the ndata_noR
total_nan_count = X_test.isna().sum().sum()

# total num of cells in ndata_noR
total_cells = X_test.size

# proportion of NaN
NaN_proportion = total_nan_count / total_cells

print('NaN_proportions:',NaN_proportion)


NaN_proportions: 0.04583333333333333


In [9]:
# 1.35% missingness

Xy_Yes_No_md = midas_impl(Xy_Yes_No, col_to_scale, 250)

Size index: [5, 3, 2, 4, 2, 6, 10, 11, 4, 4, 5, 12, 2]

Computation graph constructed

Metal device set to: Apple M1 Pro
Model initialised

Epoch: 0 , loss: 40.60445976257324
Epoch: 1 , loss: 79.321044921875
Epoch: 2 , loss: 113.20489120483398
Epoch: 3 , loss: 107.45294189453125
Epoch: 4 , loss: 71.3995590209961
Epoch: 5 , loss: 62.53718185424805
Epoch: 6 , loss: 64.57382202148438
Epoch: 7 , loss: 50.14341163635254
Epoch: 8 , loss: 42.63375282287598
Epoch: 9 , loss: 41.69715690612793
Epoch: 10 , loss: 39.88266372680664
Epoch: 11 , loss: 44.27877616882324
Epoch: 12 , loss: 35.453474044799805
Epoch: 13 , loss: 40.044307708740234
Epoch: 14 , loss: 38.9892578125
Epoch: 15 , loss: 37.579400062561035
Epoch: 16 , loss: 38.280717849731445
Epoch: 17 , loss: 39.939788818359375
Epoch: 18 , loss: 34.82228660583496
Epoch: 19 , loss: 36.479814529418945
Epoch: 20 , loss: 32.48378372192383
Epoch: 21 , loss: 37.84461784362793
Epoch: 22 , loss: 38.162864685058594
Epoch: 23 , loss: 33.08389091491699
Epoc

In [10]:
Xy_Yes_No_md

,Age_at_Surgery,Race,Surgery,Smoker,BMI,NAC,cT,pT,cT_or_pT,pN,Bx_Histology,Histology,Sample_weight_g_tumor,Tumor_digest_count_primary_tumor,Number_of_fragments_plated_tumor,OverallTILGrowth
0,69.0,1.0,2.0,0.0,19.700000,0.0,4.0,5.0,5.0,0.0,1.0,1.0,4.8900,120000000.0,24.0,1.0
1,78.0,1.0,1.0,1.0,31.770000,0.0,10.0,0.0,10.0,0.0,1.0,0.0,0.4400,225000.0,8.0,1.0
2,81.0,1.0,2.0,0.0,28.830000,0.0,1.0,6.0,1.0,0.0,1.0,1.0,0.0300,200000.0,4.0,0.0
3,74.0,1.0,2.0,1.0,44.290000,0.0,1.0,0.0,1.0,0.0,1.0,1.0,2.1100,30300000.0,18.0,0.0
4,57.0,0.0,1.0,3.0,47.140000,0.0,7.0,6.0,6.0,1.0,1.0,1.0,1.0000,23600000.0,18.0,1.0
5,67.0,1.0,1.0,1.0,21.440000,1.0,9.0,11.0,11.0,0.0,1.0,7.0,1.4400,30600000.0,24.0,1.0
6,77.0,1.0,1.0,2.0,22.200000,0.0,4.0,6.0,4.0,0.0,4.0,4.0,0.0000,2900000.0,8.0,1.0
7,69.0,1.0,1.0,1.0,26.910000,1.0,4.0,9.0,9.0,0.0,1.0,4.0,1.7000,4910000.0,3.0,1.0
8,82.0,1.0,1.0,1.0,27.130000,0.0,4.0,3.0,4.0,0.0,1.0,4.0,0.6500,1310000.0,24.0,1.0
9,91.0,1.0,1.0,0.0,24.540000,0.0,7.0,9.0,7.0,3.0,6.0,11.0,0.3400,1980000.0,18.0,1.0


In [11]:
# save to csv file

# 
Xy_Yes_No_md.to_csv('data/Xy_test_Yes_No_md.csv', index=False)

In [12]:
# compare 

Xy_Yes_No.compare(Xy_Yes_No_md, keep_shape=True, keep_equal=True)



Age_at_Surgery       Race       Surgery       Smoker          BMI  \
             self other self other    self other   self other   self   
0            69.0  69.0  1.0   1.0     2.0   2.0    0.0   0.0  19.70   
1            78.0  78.0  1.0   1.0     1.0   1.0    1.0   1.0  31.77   
2            81.0  81.0  1.0   1.0     2.0   2.0    0.0   0.0  28.83   
3            74.0  74.0  1.0   1.0     2.0   2.0    1.0   1.0  44.29   
4            57.0  57.0  0.0   0.0     1.0   1.0    3.0   3.0  47.14   
5            67.0  67.0  1.0   1.0     1.0   1.0    1.0   1.0  21.44   
6            77.0  77.0  1.0   1.0     1.0   1.0    2.0   2.0  22.20   
7            69.0  69.0  1.0   1.0     1.0   1.0    1.0   1.0  26.91   
8            82.0  82.0  1.0   1.0     1.0   1.0    1.0   1.0  27.13   
9            91.0  91.0  1.0   1.0     1.0   1.0    0.0   0.0  24.54   
10           79.0  79.0  3.0   3.0     2.0   2.0    0.0   0.0  28.86   
11           76.0  76.0  1.0   1.0     1.0   1.0    1.0   1.0  28.15   
12           79.0  79.0  1.0   1.0     1.0   1.0    1.0   1.0  27.13   
13           75.0  75.0  1.0   1.0     2.0   2.0    1.0   1.0  38.52   
14           81.0  81.0  1.0   1.0     1.0   1.0    1.0   1.0  26.80   
15           90.0  90.0  1.0   1.0     1.0   1.0    1.0   1.0  24.27   
16           54.0  54.0  1.0   1.0     1.0   1.0    2.0   2.0  22.65   
17           73.0  73.0  1.0   1.0     2.0   2.0    0.0   0.0  29.00   
18           79.0  79.0  1.0   1.0     1.0   1.0    1.0   1.0  28.10   
19           79.0  79.0  1.0   1.0     1.0   1.0    1.0   1.0  26.71   
20           47.0  47.0  1.0   1.0     2.0   2.0    0.0   0.0  22.52   
21           75.0  75.0  1.0   1.0     1.0   1.0    1.0   1.0  25.36   
22           82.0  82.0  1.0   1.0     1.0   1.0    1.0   1.0    NaN   
23           61.0  61.0  1.0   1.0     1.0   1.0    2.0   2.0  26.28   
24           78.0  78.0  1.0   1.0     1.0   1.0    1.0   1.0  26.33   
25           63.0  63.0  1.0   1.0     2.0   2.0    0.0   0.0  33.22   
26           57.0  57.0  1.0   1.0     2.0   2.0    1.0   1.0  36.75   
27           63.0  63.0  1.0   1.0     1.0   1.0    1.0   1.0  49.24   
28           69.0  69.0  1.0   1.0     1.0   1.0    2.0   2.0  28.95   
29           75.0  75.0  1.0   1.0     1.0   1.0    1.0   1.0  37.00   
30           66.0  66.0  1.0   1.0     1.0   1.0    1.0   1.0  28.69   
31           70.0  70.0  1.0   1.0     2.0   2.0    3.0   3.0  30.67   

               NAC          cT          pT       cT_or_pT         pN        \
        other self other  self other  self other     self other self other   
0   19.700000  0.0   0.0   4.0   4.0   5.0   5.0      5.0   5.0  NaN   0.0   
1   31.770000  0.0   0.0  10.0  10.0   0.0   0.0     10.0  10.0  0.0   0.0   
2   28.830000  0.0   0.0   1.0   1.0   NaN   6.0      1.0   1.0  NaN   0.0   
3   44.290000  0.0   0.0   1.0   1.0   NaN   0.0      1.0   1.0  NaN   0.0   
4   47.140000  NaN   0.0   NaN   7.0   6.0   6.0      6.0   6.0  1.0   1.0   
5   21.440000  1.0   1.0   9.0   9.0  11.0  11.0     11.0  11.0  0.0   0.0   
6   22.200000  0.0   0.0   4.0   4.0   6.0   6.0      4.0   4.0  0.0   0.0   
7   26.910000  1.0   1.0   4.0   4.0   9.0   9.0      9.0   9.0  0.0   0.0   
8   27.130000  0.0   0.0   4.0   4.0   3.0   3.0      4.0   4.0  0.0   0.0   
9   24.540000  0.0   0.0   7.0   7.0   9.0   9.0      7.0   7.0  3.0   3.0   
10  28.860000  0.0   0.0   3.0   3.0   NaN   1.0      3.0   3.0  NaN   0.0   
11  28.150000  1.0   1.0   4.0   4.0  11.0  11.0      4.0   4.0  0.0   0.0   
12  27.130000  1.0   1.0   4.0   4.0   2.0   2.0      4.0   4.0  0.0   0.0   
13  38.520000  0.0   0.0   1.0   1.0   NaN   6.0      1.0   1.0  NaN   0.0   
14  26.800000  0.0   0.0   7.0   7.0   6.0   6.0      7.0   7.0  0.0   0.0   
15  24.270000  1.0   1.0   3.0   3.0  11.0  11.0     11.0  11.0  2.0   2.0   
16  22.650000  0.0   0.0   4.0   4.0   5.0   5.0      4.0   4.0  0.0   0.0   
17  29.000000  0.0   0.0   3.0   3.0   NaN   1.0      3.0   3.0  NaN   0